# MiniCPM-V 4.7 1B — image Q&A, OCR, math, and a short video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/MiniCPM-V-4.7-1B-vision.ipynb)

[`openbmb/MiniCPM-V-4.7-1B`](https://huggingface.co/openbmb/MiniCPM-V-4.7-1B) is OpenBMB's 1B vision-language model (Apache 2.0): a Qwen3.5 text backbone plus a SigLIP vision tower with early token compression. One checkpoint does single-image questions, OCR, and video. `downsample_mode="16x"` merges visual tokens; `"4x"` keeps more detail and costs more memory. The Hub checkpoint is about 3.1 GB.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The card's `torch_dtype="auto"` loads bfloat16, and its commented FlashAttention 2 path does not run on a T4. This notebook loads **float16** and **SDPA**. Image questions sit around 6–8 GB. The video cell samples 16 frames instead of the card's 128 so the same 16 GB card can finish; raise `max_num_frames` on a larger GPU. Video decoding uses PyAV (`av`), which is the card's workaround when `torchcodec` cannot load.

**Links:** [model card](https://huggingface.co/openbmb/MiniCPM-V-4.7-1B) · [cookbook](https://github.com/OpenSQZ/MiniCPM-V-CookBook/blob/main/docs/pages/v4.7/overview.md) · [GGUF](https://huggingface.co/openbmb/MiniCPM-V-4.7-1B-gguf)


In [ ]:
%pip install -q "transformers==5.20.1" torchvision "av==16.1.0" pillow

# transformers>=5.18.0 registers MiniCPMV4_7ForConditionalGeneration, so AutoModel
# loads it with no trust_remote_code. av replaces torchcodec for video, as the card describes.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.


In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.18.0"), (
    f"transformers {transformers.__version__} is loaded; MiniCPM-V 4.7 needs 5.18+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 1. Load MiniCPM-V 4.7 1B

`AutoModelForImageTextToText` and `AutoProcessor`, as on the model card. Instruct-mode sampling below is the card's set: temperature 0.7, top_p 0.95, top_k 100, repetition penalty 1.03. `downsample_mode` has to be passed both to `apply_chat_template` and to `generate`.


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "openbmb/MiniCPM-V-4.7-1B"
DOWNSAMPLE = "16x"  # "4x" keeps more visual tokens; 16x is the card's efficient default


def load_vlm(attn):
    return AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        attn_implementation=attn,
        device_map="auto",
    ).eval()


try:
    model = load_vlm("sdpa")
    print("attention: sdpa")
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = load_vlm("eager")
    print("attention: eager")

processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


def answer(messages, max_new_tokens, **template_kwargs):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        downsample_mode=DOWNSAMPLE,
        enable_thinking=False,
        **template_kwargs,
    )
    inputs = inputs.to(model.device)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=model.dtype)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            downsample_mode=DOWNSAMPLE,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            top_k=100,
            min_p=0.0,
            presence_penalty=0,
            repetition_penalty=1.03,
        )
    trimmed = generated[0, inputs["input_ids"].shape[1] :]
    text = processor.decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    return text


## 2. Image question

The photograph is the card's refraction example. `max_slice_nums=9` is the processor default. The card suggests 36 when you want finer detail and have the memory for it.


In [ ]:
from IPython.display import Image, display

REFRACT = "https://huggingface.co/datasets/openbmb/DemoCase/resolve/main/refract.png"
display(Image(url=REFRACT, width=480))
print(answer(
    [{
        "role": "user",
        "content": [
            {"type": "image", "url": REFRACT},
            {"type": "text", "text": "What causes this phenomenon?"},
        ],
    }],
    max_new_tokens=256,
    max_slice_nums=9,
))


## 3. OCR — read a receipt


In [ ]:
RECEIPT = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
display(Image(url=RECEIPT, width=420))
print(answer(
    [{
        "role": "user",
        "content": [
            {"type": "image", "url": RECEIPT},
            {"type": "text", "text": "What is the text in the picture?"},
        ],
    }],
    max_new_tokens=400,
    max_slice_nums=9,
))


## 4. Math written as an image

The equation is drawn in the notebook so the model has to read it, not repeat a text prompt.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage

fig, ax = plt.subplots(figsize=(7, 1.6))
ax.axis("off")
ax.text(0.5, 0.5, r"$\int_0^3 x^2\,dx$", fontsize=32, ha="center", va="center")
fig.savefig("integral.png", bbox_inches="tight", dpi=160)
plt.close(fig)
integral = PILImage.open("integral.png").convert("RGB")
display(integral)
print(answer(
    [{
        "role": "user",
        "content": [
            {"type": "image", "image": integral},
            {"type": "text", "text": "Evaluate the integral in the image. Show the antiderivative and the numeric result."},
        ],
    }],
    max_new_tokens=300,
    max_slice_nums=4,
))


## 5. A short video

The clip is the card's football example. Sampling stops at 16 frames (`max_num_frames=128` on the card is for a larger GPU). A few frames are shown first so the clip is visible before the caption.


In [ ]:
import av
import matplotlib.pyplot as plt
import requests

VIDEO_URL = "https://huggingface.co/datasets/openbmb/DemoCase/resolve/main/football.mp4"
video_path = "football.mp4"
response = requests.get(VIDEO_URL, timeout=180)
response.raise_for_status()
open(video_path, "wb").write(response.content)

container = av.open(video_path)
stream = container.streams.video[0]
wanted = 4
step = max(int(stream.frames or 1) // wanted, 1)
shown = 0
fig, axes = plt.subplots(1, wanted, figsize=(12, 3))
for index, frame in enumerate(container.decode(stream)):
    if index % step:
        continue
    axes[shown].imshow(frame.to_image())
    axes[shown].set_axis_off()
    shown += 1
    if shown == wanted:
        break
container.close()
fig.suptitle("frames from the clip")
fig.tight_layout()
plt.show()

print(answer(
    [{
        "role": "user",
        "content": [
            {"type": "video", "url": VIDEO_URL},
            {"type": "text", "text": "Describe this video. Follow the timeline: main actions and any on-screen text."},
        ],
    }],
    max_new_tokens=300,
    max_num_frames=16,
    stack_frames=1,
    max_slice_nums=1,
    use_image_id=False,
))


## 6. Optional — llama.cpp GGUF

The official GGUF repo is [`openbmb/MiniCPM-V-4.7-1B-gguf`](https://huggingface.co/openbmb/MiniCPM-V-4.7-1B-gguf). MiniCPM-V 4.7 support is in upstream llama.cpp (PR [#29416](https://github.com/ggml-org/llama.cpp/pull/29416), merged 2026-10-10). This cell pins the CUDA 12.8 build `b11721`, which is after that PR, and the cookbook's `Q4_K_M` + `mmproj-model-f16.gguf` command. `-ngl 99` offloads to the T4. `-fa off` keeps FlashAttention off on sm_75. `--reasoning off` skips the thinking trace the template enables by default.

Skip this section if the transformers demo above is enough. It downloads about 1.8 GB of GGUF plus the llama.cpp archive, and it reuses `integral.png` from the math section.


In [ ]:
import glob
import os
import subprocess
import tarfile
import urllib.request

from huggingface_hub import hf_hub_download

LLAMA_TAG = "b11721"
LLAMA_URL = (
    "https://github.com/ggml-org/llama.cpp/releases/download/"
    f"{LLAMA_TAG}/llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
)
BIN = "/content/llama-minicpm"


def cuda_lib_dirs():
    found = []
    patterns = [
        "/usr/local/cuda*/lib64",
        "/usr/local/cuda*/targets/x86_64-linux/lib",
        "/usr/lib/x86_64-linux-gnu",
        "/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib",
    ]
    for pattern in patterns:
        found.extend(glob.glob(pattern))
    return [path for path in found if os.path.isdir(path)]


if not os.path.exists(os.path.join(BIN, "llama-mtmd-cli")):
    archive = "/tmp/llama-minicpm.tgz"
    urllib.request.urlretrieve(LLAMA_URL, archive)
    os.makedirs(BIN, exist_ok=True)
    with tarfile.open(archive) as tar:
        tar.extractall(BIN)
    # The archive has a single top-level directory.
    nested = [os.path.join(BIN, name) for name in os.listdir(BIN)]
    if len(nested) == 1 and os.path.isdir(nested[0]):
        for name in os.listdir(nested[0]):
            os.rename(os.path.join(nested[0], name), os.path.join(BIN, name))

libs = cuda_lib_dirs()
if not any(os.path.exists(os.path.join(path, "libcudart.so.12")) for path in libs + [BIN]):
    raise SystemExit(
        "The CUDA 12.8 llama.cpp build needs libcudart.so.12. "
        "On Colab it lives under /usr/local/cuda/lib64. None was found."
    )

ENV = os.environ.copy()
ENV["LD_LIBRARY_PATH"] = ":".join([BIN, *libs, ENV.get("LD_LIBRARY_PATH", "")])

llm = hf_hub_download("openbmb/MiniCPM-V-4.7-1B-gguf", "MiniCPM-V-4_7-1B-Q4_K_M.gguf")
mmproj = hf_hub_download("openbmb/MiniCPM-V-4.7-1B-gguf", "mmproj-model-f16.gguf")
print(llm)
print(mmproj)

cmd = [
    os.path.join(BIN, "llama-mtmd-cli"),
    "-m", llm,
    "--mmproj", mmproj,
    "-ngl", "99",
    "-fa", "off",
    "-c", "8192",
    "--temp", "0.7",
    "--top-p", "0.8",
    "--top-k", "100",
    "--repeat-penalty", "1.05",
    "--reasoning", "off",
    "--image", "integral.png",
    "-p", "What equation is in the image, and what is its value?",
    "-n", "200",
]
print(subprocess.run(cmd, env=ENV, stdin=subprocess.DEVNULL, check=False).returncode)
